# Notebook 03 — Evaluación del Modelo
**Objetivo:** Medir mejora del fine-tuning vs modelo base en tareas de contratos
**Entradas:** adapter en Drive, 100 ejemplos del conjunto de prueba
**Salidas:** `/content/drive/MyDrive/agente-contratos-cto/metricas/resultados_evaluacion.json`
**Tiempo estimado:** ~20 minutos

In [ ]:
!pip install -q torch==2.3.0 transformers==4.44.0 peft==0.12.0 bitsandbytes==0.43.3 datasets==2.21.0 evaluate==0.4.3 rouge-score==0.1.2 bert-score==0.3.13 accelerate==0.33.0 sentencepiece==0.2.0

In [ ]:
import torch
from google.colab import drive
import os

# --- Verificar disponibilidad de GPU ---
if not torch.cuda.is_available():
    raise RuntimeError(
        "No se detectó GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno "
        "de ejecución y selecciona GPU T4."
    )

nombre_gpu = torch.cuda.get_device_name(0)
memoria_gpu = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU detectada: {nombre_gpu}")
print(f"Memoria total: {memoria_gpu:.1f} GB")

# --- Montar Google Drive ---
drive.mount("/content/drive")

# --- Verificar que el adapter existe ---
RUTA_ADAPTER = "/content/drive/MyDrive/agente-contratos-cto/adapter/"

if not os.path.isdir(RUTA_ADAPTER):
    raise FileNotFoundError(
        f"No se encontró el directorio del adapter en: {RUTA_ADAPTER}\n"
        "Asegúrate de haber ejecutado el notebook 02 de entrenamiento primero."
    )

archivos_adapter = os.listdir(RUTA_ADAPTER)
print(f"\nAdapter encontrado en: {RUTA_ADAPTER}")
print(f"Archivos del adapter: {archivos_adapter}")

In [ ]:
import json
from datasets import Dataset

# --- Cargar conjunto de datos de prueba ---
RUTA_DATASET = "/content/drive/MyDrive/agente-contratos-cto/dataset/contratos_sft.jsonl"

if not os.path.isfile(RUTA_DATASET):
    raise FileNotFoundError(
        f"No se encontró el dataset en: {RUTA_DATASET}\n"
        "Asegúrate de haber ejecutado el notebook 01 de generación de datos."
    )

# Leer todas las líneas del archivo JSONL
todos_los_ejemplos: list[dict] = []
with open(RUTA_DATASET, "r", encoding="utf-8") as archivo:
    for linea in archivo:
        linea = linea.strip()
        if linea:
            todos_los_ejemplos.append(json.loads(linea))

print(f"Total de ejemplos en el dataset: {len(todos_los_ejemplos)}")

# Usar el mismo split que notebook 02 (90/10 con seed=42 via datasets)
dataset_completo = Dataset.from_list(todos_los_ejemplos)
particiones = dataset_completo.train_test_split(test_size=0.1, seed=42)
conjunto_prueba: list[dict] = list(particiones["test"])

print(f"Ejemplos en el conjunto de prueba: {len(conjunto_prueba)}")

# Mostrar un ejemplo para verificar la estructura
print(f"\nEjemplo de estructura del dataset:")
print(json.dumps(conjunto_prueba[0], indent=2, ensure_ascii=False)[:500])

## Carga de Modelos

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- Configuración del modelo base ---
NOMBRE_MODELO_BASE: str = "Qwen/Qwen2.5-7B-Instruct"

# Configuración de cuantización a 4 bits
configuracion_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --- Cargar tokenizador ---
tokenizador = AutoTokenizer.from_pretrained(
    NOMBRE_MODELO_BASE,
    trust_remote_code=True,
)
if tokenizador.pad_token is None:
    tokenizador.pad_token = tokenizador.eos_token

print("Cargando modelo base (sin adapter) para línea base...")

# --- Cargar modelo base SIN adapter ---
modelo_base = AutoModelForCausalLM.from_pretrained(
    NOMBRE_MODELO_BASE,
    quantization_config=configuracion_4bit,
    device_map="auto",
    trust_remote_code=True,
)
modelo_base.eval()

print(f"Modelo base cargado: {NOMBRE_MODELO_BASE}")
print(f"Dispositivo: {modelo_base.device}")

In [ ]:
from peft import PeftModel

print("Cargando modelo fine-tuned (base + adapter LoRA)...")

# --- Cargar modelo base con cuantización para aplicar el adapter ---
modelo_ft_base = AutoModelForCausalLM.from_pretrained(
    NOMBRE_MODELO_BASE,
    quantization_config=configuracion_4bit,
    device_map="auto",
    trust_remote_code=True,
)

# --- Aplicar el adapter LoRA desde Drive ---
modelo_fine_tuned = PeftModel.from_pretrained(
    modelo_ft_base,
    RUTA_ADAPTER,
)
modelo_fine_tuned.eval()

print(f"Modelo fine-tuned cargado con adapter desde: {RUTA_ADAPTER}")
print(f"Dispositivo: {modelo_fine_tuned.device}")

## Función de Generación de Respuestas

In [ ]:
def generar_respuesta(
    modelo,
    tokenizador,
    instruccion: str,
    entrada: str,
    max_tokens: int = 512,
) -> str:
    """Genera una respuesta utilizando el modelo y tokenizador proporcionados.

    Formatea la entrada usando la plantilla de chat de Qwen2.5 con un mensaje
    de sistema especializado en análisis de contratos tecnológicos.

    Args:
        modelo: Modelo de lenguaje (base o fine-tuned) para generar la respuesta.
        tokenizador: Tokenizador asociado al modelo.
        instruccion: Instrucción o tarea que el modelo debe realizar.
        entrada: Texto de entrada o contexto del contrato a analizar.
        max_tokens: Número máximo de tokens nuevos a generar. Por defecto 512.

    Returns:
        Texto de la respuesta generada por el modelo.
    """
    # Mensaje de sistema para el agente de contratos
    mensaje_sistema: str = (
        "Eres un agente experto en análisis de contratos tecnológicos para CTOs."
    )

    # Construir el contenido del usuario combinando instrucción y entrada
    contenido_usuario: str = f"{instruccion}\n\n{entrada}" if entrada else instruccion

    # Formatear usando la plantilla de chat de Qwen2.5
    mensajes: list[dict] = [
        {"role": "system", "content": mensaje_sistema},
        {"role": "user", "content": contenido_usuario},
    ]

    texto_formateado: str = tokenizador.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenizar la entrada
    entradas_modelo = tokenizador(
        texto_formateado,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(modelo.device)

    longitud_entrada: int = entradas_modelo["input_ids"].shape[1]

    # Generar respuesta
    with torch.no_grad():
        salida = modelo.generate(
            **entradas_modelo,
            max_new_tokens=max_tokens,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            pad_token_id=tokenizador.pad_token_id,
        )

    # Decodificar solo los tokens generados (sin la entrada)
    tokens_generados = salida[0][longitud_entrada:]
    respuesta: str = tokenizador.decode(tokens_generados, skip_special_tokens=True)

    return respuesta.strip()


# --- Prueba rápida de la función ---
print("Probando función de generación con modelo base...")
respuesta_prueba = generar_respuesta(
    modelo=modelo_base,
    tokenizador=tokenizador,
    instruccion="Resume la siguiente cláusula.",
    entrada="El proveedor se compromete a entregar el software en un plazo de 90 días.",
    max_tokens=100,
)
print(f"Respuesta de prueba: {respuesta_prueba[:200]}")

## Generación de Respuestas para Evaluación

In [ ]:
from tqdm.notebook import tqdm

# --- Generar respuestas de ambos modelos para los 100 ejemplos de prueba ---
respuestas_base: list[str] = []
respuestas_fine_tuned: list[str] = []
referencias: list[str] = []
instrucciones: list[str] = []
entradas: list[str] = []

print("Generando respuestas para los 100 ejemplos de prueba...")
print("Esto puede tardar varios minutos.\n")

for i, ejemplo in enumerate(tqdm(conjunto_prueba, desc="Evaluando ejemplos")):
    instruccion_actual: str = ejemplo.get("instruccion", "")
    entrada_actual: str = ejemplo.get("entrada", "")
    referencia_actual: str = ejemplo.get("salida", "")

    instrucciones.append(instruccion_actual)
    entradas.append(entrada_actual)
    referencias.append(referencia_actual)

    # Generar respuesta con modelo base
    respuesta_b: str = generar_respuesta(
        modelo=modelo_base,
        tokenizador=tokenizador,
        instruccion=instruccion_actual,
        entrada=entrada_actual,
    )
    respuestas_base.append(respuesta_b)

    # Generar respuesta con modelo fine-tuned
    respuesta_ft: str = generar_respuesta(
        modelo=modelo_fine_tuned,
        tokenizador=tokenizador,
        instruccion=instruccion_actual,
        entrada=entrada_actual,
    )
    respuestas_fine_tuned.append(respuesta_ft)

    # Mostrar progreso cada 10 ejemplos
    if (i + 1) % 10 == 0:
        print(f"  Procesados {i + 1}/100 ejemplos")

print(f"\nGeneración completada.")
print(f"  Respuestas modelo base: {len(respuestas_base)}")
print(f"  Respuestas modelo fine-tuned: {len(respuestas_fine_tuned)}")
print(f"  Referencias: {len(referencias)}")

## Cálculo de Métricas: ROUGE-L

In [ ]:
import evaluate

# --- Calcular ROUGE-L para ambos modelos ---
metrica_rouge = evaluate.load("rouge")

# ROUGE-L para el modelo base
resultados_rouge_base = metrica_rouge.compute(
    predictions=respuestas_base,
    references=referencias,
)
rouge_l_base: float = resultados_rouge_base["rougeL"]

# ROUGE-L para el modelo fine-tuned
resultados_rouge_ft = metrica_rouge.compute(
    predictions=respuestas_fine_tuned,
    references=referencias,
)
rouge_l_fine_tuned: float = resultados_rouge_ft["rougeL"]

# Calcular porcentaje de mejora
if rouge_l_base > 0:
    mejora_rouge_pct: float = ((rouge_l_fine_tuned - rouge_l_base) / rouge_l_base) * 100
else:
    mejora_rouge_pct: float = 0.0

print("=== Resultados ROUGE-L ===")
print(f"  Modelo base:       {rouge_l_base:.4f}")
print(f"  Modelo fine-tuned: {rouge_l_fine_tuned:.4f}")
print(f"  Mejora:            {mejora_rouge_pct:+.2f}%")

## Cálculo de Métricas: BERTScore

In [ ]:
# --- Calcular BERTScore F1 para ambos modelos ---
# Usar modelo BERT en español para mayor precisión
MODELO_BERT_ESPANOL: str = "dccuchile/bert-base-spanish-wwm-cased"

metrica_bertscore = evaluate.load("bertscore")

print("Calculando BERTScore para el modelo base...")
resultados_bert_base = metrica_bertscore.compute(
    predictions=respuestas_base,
    references=referencias,
    model_type=MODELO_BERT_ESPANOL,
    batch_size=16,
)
# Promediar los scores F1 de todos los ejemplos
bert_f1_base: float = sum(resultados_bert_base["f1"]) / len(resultados_bert_base["f1"])

print("Calculando BERTScore para el modelo fine-tuned...")
resultados_bert_ft = metrica_bertscore.compute(
    predictions=respuestas_fine_tuned,
    references=referencias,
    model_type=MODELO_BERT_ESPANOL,
    batch_size=16,
)
bert_f1_fine_tuned: float = sum(resultados_bert_ft["f1"]) / len(resultados_bert_ft["f1"])

# Calcular porcentaje de mejora
if bert_f1_base > 0:
    mejora_bert_pct: float = ((bert_f1_fine_tuned - bert_f1_base) / bert_f1_base) * 100
else:
    mejora_bert_pct: float = 0.0

print("\n=== Resultados BERTScore F1 ===")
print(f"  Modelo base:       {bert_f1_base:.4f}")
print(f"  Modelo fine-tuned: {bert_f1_fine_tuned:.4f}")
print(f"  Mejora:            {mejora_bert_pct:+.2f}%")

## Tabla Comparativa de Resultados

In [ ]:
# --- Tabla comparativa de métricas ---
print("=" * 70)
print("          TABLA COMPARATIVA DE RESULTADOS")
print("=" * 70)
print(f"{'Métrica':<20} {'Modelo Base':>15} {'Fine-Tuned':>15} {'Mejora %':>15}")
print("-" * 70)
print(
    f"{'ROUGE-L':<20} {rouge_l_base:>15.4f} {rouge_l_fine_tuned:>15.4f} "
    f"{mejora_rouge_pct:>+14.2f}%"
)
print(
    f"{'BERTScore F1':<20} {bert_f1_base:>15.4f} {bert_f1_fine_tuned:>15.4f} "
    f"{mejora_bert_pct:>+14.2f}%"
)
print("=" * 70)
print(f"\nNúmero de ejemplos evaluados: {len(conjunto_prueba)}")
print(f"Modelo base: {NOMBRE_MODELO_BASE}")
print(f"Adapter: {RUTA_ADAPTER}")

## Evaluación Cualitativa

In [ ]:
# --- Evaluación cualitativa: 5 ejemplos por tipo de tarea (20 total) ---

# Los 4 tipos de tarea definidos en el notebook 01
TIPOS_TAREA: list[str] = [
    "analizar_clausula",
    "comparar_propuestas",
    "alerta_renovacion",
    "estrategia_negociacion",
]


def mostrar_ejemplo_cualitativo(
    numero: int,
    tipo_tarea: str,
    instruccion: str,
    entrada: str,
    referencia: str,
    respuesta_base: str,
    respuesta_ft: str,
) -> None:
    """Muestra un ejemplo cualitativo formateado para comparación visual.

    Imprime de forma estructurada la entrada, la salida esperada y las
    respuestas de ambos modelos para facilitar la evaluación cualitativa.

    Args:
        numero: Número secuencial del ejemplo.
        tipo_tarea: Tipo de tarea del dataset.
        instruccion: Instrucción original del ejemplo.
        entrada: Texto de entrada del ejemplo.
        referencia: Respuesta esperada (referencia).
        respuesta_base: Respuesta generada por el modelo base.
        respuesta_ft: Respuesta generada por el modelo fine-tuned.
    """
    limite_chars: int = 300

    print(f"\n{'=' * 80}")
    print(f"EJEMPLO {numero} | Tipo de tarea: {tipo_tarea}")
    print(f"{'=' * 80}")

    print(f"\n--- INSTRUCCIÓN ---")
    print(instruccion[:limite_chars])

    print(f"\n--- ENTRADA ---")
    print(entrada[:limite_chars] + ("..." if len(entrada) > limite_chars else ""))

    print(f"\n--- SALIDA ESPERADA ---")
    print(referencia[:limite_chars] + ("..." if len(referencia) > limite_chars else ""))

    print(f"\n--- RESPUESTA MODELO BASE ---")
    print(respuesta_base[:limite_chars] + ("..." if len(respuesta_base) > limite_chars else ""))

    print(f"\n--- RESPUESTA MODELO FINE-TUNED ---")
    print(respuesta_ft[:limite_chars] + ("..." if len(respuesta_ft) > limite_chars else ""))


# --- Agrupar ejemplos por tipo de tarea usando el campo del dataset ---
ejemplos_por_tipo: dict[str, list[int]] = {tipo: [] for tipo in TIPOS_TAREA}
for idx, ejemplo in enumerate(conjunto_prueba):
    tipo: str = ejemplo.get("tipo_tarea", "otro")
    if tipo in ejemplos_por_tipo:
        ejemplos_por_tipo[tipo].append(idx)

print("Distribución de tipos de tarea en el conjunto de prueba:")
for tipo_t, indices_t in ejemplos_por_tipo.items():
    print(f"  {tipo_t}: {len(indices_t)} ejemplos")

# --- Seleccionar 5 ejemplos por tipo (20 total) ---
ejemplos_cualitativos: list[dict] = []
contador_ejemplo: int = 1
EJEMPLOS_POR_TIPO: int = 5

print(f"\n{'#' * 80}")
print(f"    EVALUACIÓN CUALITATIVA: 5 ejemplos por tipo de tarea")
print(f"{'#' * 80}")

for tipo_tarea in TIPOS_TAREA:
    indices = ejemplos_por_tipo[tipo_tarea]
    indices_seleccionados: list[int] = indices[:EJEMPLOS_POR_TIPO]

    for idx in indices_seleccionados:
        mostrar_ejemplo_cualitativo(
            numero=contador_ejemplo,
            tipo_tarea=tipo_tarea,
            instruccion=instrucciones[idx],
            entrada=entradas[idx],
            referencia=referencias[idx],
            respuesta_base=respuestas_base[idx],
            respuesta_ft=respuestas_fine_tuned[idx],
        )

        ejemplos_cualitativos.append({
            "numero": contador_ejemplo,
            "tipo_tarea": tipo_tarea,
            "instruccion": instrucciones[idx],
            "entrada": entradas[idx],
            "salida_esperada": referencias[idx],
            "respuesta_modelo_base": respuestas_base[idx],
            "respuesta_modelo_fine_tuned": respuestas_fine_tuned[idx],
        })
        contador_ejemplo += 1

print(f"\nTotal de ejemplos cualitativos seleccionados: {len(ejemplos_cualitativos)}")

## Guardar Resultados

In [ ]:
from datetime import datetime

# --- Construir diccionario de resultados ---
resultados: dict = {
    "fecha_evaluacion": datetime.now().strftime("%Y-%m-%d"),
    "modelo_base": "Qwen/Qwen2.5-7B-Instruct",
    "n_ejemplos_evaluados": 100,
    "metricas": {
        "rouge_l": {
            "base": round(rouge_l_base, 4),
            "fine_tuned": round(rouge_l_fine_tuned, 4),
            "mejora_pct": round(mejora_rouge_pct, 2),
        },
        "bert_score_f1": {
            "base": round(bert_f1_base, 4),
            "fine_tuned": round(bert_f1_fine_tuned, 4),
            "mejora_pct": round(mejora_bert_pct, 2),
        },
    },
    "ejemplos_cualitativos": ejemplos_cualitativos,
}

# --- Crear directorio de métricas si no existe ---
RUTA_METRICAS: str = "/content/drive/MyDrive/agente-contratos-cto/metricas"
os.makedirs(RUTA_METRICAS, exist_ok=True)

# --- Guardar resultados en JSON ---
RUTA_RESULTADOS: str = os.path.join(RUTA_METRICAS, "resultados_evaluacion.json")

with open(RUTA_RESULTADOS, "w", encoding="utf-8") as archivo_json:
    json.dump(resultados, archivo_json, indent=2, ensure_ascii=False)

print(f"Resultados guardados exitosamente en:")
print(f"  {RUTA_RESULTADOS}")
print(f"\nResumen final:")
print(f"  Fecha de evaluación: {resultados['fecha_evaluacion']}")
print(f"  Modelo base: {resultados['modelo_base']}")
print(f"  Ejemplos evaluados: {resultados['n_ejemplos_evaluados']}")
print(f"  ROUGE-L  - Base: {rouge_l_base:.4f} | FT: {rouge_l_fine_tuned:.4f} | Mejora: {mejora_rouge_pct:+.2f}%")
print(f"  BERTScore - Base: {bert_f1_base:.4f} | FT: {bert_f1_fine_tuned:.4f} | Mejora: {mejora_bert_pct:+.2f}%")
print(f"  Ejemplos cualitativos: {len(ejemplos_cualitativos)}")